<a href="https://colab.research.google.com/github/angelms2003/FernandezMartinezPolo-EML-RL/blob/main/Entornos_Complejos/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

    Authors: David Fernández Expósito
             Ángel Martínez Sánchez
             Javier Polo Gambín
    Emails: dfernandezexposito@um.es
            angel.martinezs@um.es
            javier.polog@um.es
    Date: 2026/02/14

In [ ]:
!git clone https://github.com/angelms2003/FernandezMartinezPolo-EML-RL.git

In [ ]:
!cd FernandezMartinezPolo-EML-RL/

In [ ]:
#@title Importamos todas las clases y funciones

import sys

# Añadir los directorio fuentes al path de Python
sys.path.append('/content/FernandezMartinezPolo-EML-RL/Entornos_Complejos/src')


# Verificar que se han añadido correctamente
print(sys.path)

import numpy as np
from typing import List

from algorithms import Algorithm, EpsilonGreedy, EpsilonDecay, UCB1, UCB2, Softmax
from arms import ArmNormal, Bandit, ArmBernoulli, ArmBinomial
from plotting import plot_average_rewards, plot_optimal_selections, plot_regret, plot_arm_statistics

Este notebook sirve como enlace para poder acceder a todos los notebooks que hemos desarrollado. Los enlaces se muestran a continuación:

- [📓 Notebook Monte Carlo On Policy](https://colab.research.google.com/github/angelms2003/FernandezMartinezPolo-EML-RL/blob/main/Entornos_Complejos/MonteCarlo_OnPolicy.ipynb)
- [📓 Notebook Monte Carlo Off Policy](https://colab.research.google.com/github/angelms2003/FernandezMartinezPolo-EML-RL/blob/main/Entornos_Complejos/MonteCarlo_OffPolicy.ipynb)
- [📓 Notebook SARSA](https://colab.research.google.com/github/angelms2003/FernandezMartinezPolo-EML-RL/blob/main/Entornos_Complejos/SARSA.ipynb)
- [📓 Notebook Q Learning](https://colab.research.google.com/github/angelms2003/FernandezMartinezPolo-EML-RL/blob/main/Entornos_Complejos/Q_learning.ipynb)
- [📓 Notebook SARSA Semi-gradiente](https://colab.research.google.com/github/angelms2003/FernandezMartinezPolo-EML-RL/blob/main/Entornos_Complejos/SARSA_semi_gradiente.ipynb)
- [📓 Notebook Deep Q Learning](https://colab.research.google.com/github/angelms2003/FernandezMartinezPolo-EML-RL/blob/main/Entornos_Complejos/Deep_Q_Learning.ipynb)

# Unión de resultados para plots conjuntos (Entornos complejos y metodos tabulares)

Ahora vamos a generar algunas gráficas conjuntas de los cuatro algoritmos, para poder mostrar la comparación más fácilmente tanto en el informe escrito como en la presentación. Tomamos para la comparación la mejor opción de cada algoritmo, que fueron: 

- Monte Carlo On Policy (epsilon= 0.6, decay= True)
- Monte Carlo Off Policy (epsilon= 0.3, decay= True)
- SARSA ()
- Q Learning ()

In [ ]:
import numpy as np

# Cargar ficheros
data_exp1 = np.load('/content/FernandezMartinezPolo-EML-RL/Entornos_Complejos/data/monte_carlo_off_results.npz', allow_pickle=True)
data_exp2 = np.load('/content/FernandezMartinezPolo-EML-RL/Entornos_Complejos/data/monte_carlo_results.npz', allow_pickle=True)

# Recuperar diccionarios
stats_exp1 = data_exp1["dict_stats"].item()
len_exp1 = data_exp1["dict_len"].item()

stats_exp2 = data_exp2["dict_stats"].item()
len_exp2 = data_exp2["dict_len"].item()

# Convertir claves a lista para indexar
keys_exp1 = list(stats_exp1.keys())
keys_exp2 = list(stats_exp2.keys())

# Seleccionar algoritmos
alg1_key = keys_exp1[0]   # primero del primer fichero
alg2_key = keys_exp2[4]   # quinto del segundo fichero

alg1_stats = stats_exp1[alg1_key]
alg1_len = len_exp1[alg1_key]

alg2_stats = stats_exp2[alg2_key]
alg2_len = len_exp2[alg2_key]

print("Algoritmo 1:", alg1_key)
print("Algoritmo 2:", alg2_key)

In [ ]:
def draw_multiple_learning_curves(results_dict):
    """
    Representa varias curvas de entrenamiento en el mismo gráfico.

    results_dict:
        Diccionario donde:
        clave -> nombre experimento/agente
        valor -> lista con historial de métricas
    """

    first_key = next(iter(results_dict))
    x_axis = np.arange(len(results_dict[first_key]))

    fig, ax = plt.subplots(figsize=(10, 4))

    for experiment_name, history in results_dict.items():
        ax.plot(x_axis, history, label=experiment_name)

    ax.set_title("Comparativa de rendimiento")
    ax.set_xlabel("Número de episodio")
    ax.set_ylabel("Valor medio")
    ax.legend()

    ax.grid()
    plt.show()


def compute_running_mean(series, window):
    """
    Calcula un suavizado tipo media deslizante sobre una serie temporal.
    """
    kernel = np.full(window, 1.0 / window)
    return np.convolve(series, kernel, mode="valid")


def draw_episode_length_trend(length_history):
    """
    Visualiza la evolución del tamaño de los episodios junto con
    una versión suavizada para observar la tendencia global.
    """

    fig, ax = plt.subplots(figsize=(10, 4))

    # Señal original (transparente)
    ax.plot(length_history, alpha=0.25, color="forestgreen")

    # Tendencia suavizada
    smoothing_window = 100
    smoothed = compute_running_mean(length_history, smoothing_window)

    ax.plot(
        np.arange(len(smoothed)),
        smoothed,
        linewidth=2,
        color="darkgreen",
        label="Media suavizada"
    )

    ax.set_title("Evolución del tamaño de episodio")
    ax.set_xlabel("Índice de episodio")
    ax.set_ylabel("Número de pasos")
    ax.legend()
    ax.grid()

    plt.show()


def draw_episode_length_comparison(length_dict):
    """
    Compara la evolución de longitud de episodio entre varios experimentos.
    """

    palette = ["darkred", "navy", "darkgreen", "purple", "orange"]

    fig, ax = plt.subplots(figsize=(10, 4))

    smoothing_window = 100

    for idx, (label, values) in enumerate(length_dict.items()):

        color = palette[idx % len(palette)]

        # Señal original
        ax.plot(values, alpha=0.2, color=color)

        # Tendencia suavizada
        smoothed = compute_running_mean(values, smoothing_window)
        ax.plot(
            np.arange(len(smoothed)),
            smoothed,
            linewidth=2,
            color=color,
            label=label
        )

    ax.set_title("Comparativa de longitudes de episodio")
    ax.set_xlabel("Índice de episodio")
    ax.set_ylabel("Número de pasos")
    ax.legend()
    ax.grid()

    plt.show()

In [ ]:
import matplotlib.pyplot as plt

############################################
# PLOT STATS
############################################

fig_stats, ax_stats = plt.subplots(figsize=(10,6))

ax_stats.plot(alg1_stats, label=f'Alg1 ({alg1_key})')
ax_stats.plot(alg2_stats, label=f'Alg2 ({alg2_key})')

ax_stats.set_title('Stats comparison')
ax_stats.set_xlabel('Episodes')
ax_stats.set_ylabel('Stats')
ax_stats.legend()
ax_stats.grid(True)

plt.tight_layout()
plt.savefig('stats_comparison.png')
plt.show()


############################################
# PLOT EPISODE LENGTH
############################################

fig_len, ax_len = plt.subplots(figsize=(10,6))

ax_len.plot(alg1_len, label=f'Alg1 ({alg1_key})')
ax_len.plot(alg2_len, label=f'Alg2 ({alg2_key})')

ax_len.set_title('Episode length comparison')
ax_len.set_xlabel('Episodes')
ax_len.set_ylabel('Length')
ax_len.legend()
ax_len.grid(True)

plt.tight_layout()
plt.savefig('length_comparison.png')
plt.show()

Extracted data for the third experiment from all NPZ files.
